In [2]:
!pip install ollama

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import json

# Load your Stage 1 frame data (with real GPS + weather)
with open("../outputs/incidents/stage1_frames.json") as f:
    stage1_frames = json.load(f)

print(f"Loaded {len(stage1_frames)} frames")
print(stage1_frames[0])

Loaded 164 frames
{'frame_number': 0, 'timestamp_sec': 0.0, 'frame_path': '../data/extracted_frames\\frame_00000.jpg', 'gps': [13.0827, 80.2707], 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}}


In [8]:
from math import radians, sin, cos, sqrt, atan2

def gps_distance_meters(gps1, gps2):
    R = 6371000
    lat1, lon1 = radians(gps1[0]), radians(gps1[1])
    lat2, lon2 = radians(gps2[0]), radians(gps2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def verify_incidents(detections, min_conf=0.6, max_frame_gap=5, max_gps_dist=20, min_streak=3):
    detections = sorted(detections, key=lambda d: d["frame"])
    incidents = []
    current_group = []

    for det in detections:
        if det["conf"] < min_conf:
            continue
        if not current_group:
            current_group = [det]
            continue

        last = current_group[-1]
        same_class = det["class"] == last["class"]
        close_frame = (det["frame"] - last["frame"]) <= max_frame_gap
        close_gps = gps_distance_meters(det["gps"], last["gps"]) <= max_gps_dist

        if same_class and close_frame and close_gps:
            current_group.append(det)
        else:
            if len(current_group) >= min_streak:
                incidents.append(build_incident(current_group))
            current_group = [det]

    if len(current_group) >= min_streak:
        incidents.append(build_incident(current_group))

    return incidents

def build_incident(group):
    avg_conf = sum(d["conf"] for d in group) / len(group)
    return {
        "incident_id": f"RA-{group[0]['frame']:04d}",
        "anomaly": group[0]["class"],
        "status": "VERIFIED",
        "evidence_frames": len(group),
        "avg_confidence": round(avg_conf, 3),
        "gps": group[len(group)//2]["gps"],
        "frame_range": (group[0]["frame"], group[-1]["frame"])
    }

# Dummy detections (same as your earlier test)
sample_detections = [
    {"frame": 118, "class": "D40_pothole", "conf": 0.71, "gps": (13.04500, 80.21100), "bbox": [120,200,180,260]},
    {"frame": 119, "class": "D40_pothole", "conf": 0.88, "gps": (13.04502, 80.21102), "bbox": [122,198,182,258]},
    {"frame": 120, "class": "D40_pothole", "conf": 0.94, "gps": (13.04504, 80.21104), "bbox": [124,199,184,259]},
    {"frame": 121, "class": "D40_pothole", "conf": 0.92, "gps": (13.04506, 80.21106), "bbox": [125,200,185,260]},
    {"frame": 200, "class": "D00_longitudinal_crack", "conf": 0.55, "gps": (13.0600, 80.2200), "bbox": [50,50,90,55]},
]

incidents = verify_incidents(sample_detections)
print(f"Verified incidents: {len(incidents)}")
for inc in incidents:
    print(inc)

Verified incidents: 1
{'incident_id': 'RA-0118', 'anomaly': 'D40_pothole', 'status': 'VERIFIED', 'evidence_frames': 4, 'avg_confidence': 0.863, 'gps': (13.04504, 80.21104), 'frame_range': (118, 121)}


In [15]:
def context_agent(incident, weather, repetition_count=1):
    anomaly = incident["anomaly"]
    condition = weather.get("condition", "Unknown")
    is_rainy = condition in ("Rain", "Drizzle", "Thunderstorm")

    context = {
        "incident_id": incident["incident_id"],
        "anomaly": anomaly,
        "weather_condition": condition,
        "repetition_count": repetition_count,
        "conclusion": None,
        "risk_level": "Normal"
    }

    if "water" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Water accumulation likely related to current rainfall"
        elif repetition_count > 1:
            context["conclusion"] = "Possible persistent drainage issue"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Water presence unexplained by weather — needs inspection"
            context["risk_level"] = "Increased"
    elif "pothole" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Pothole with active rain — increased vehicle hazard"
            context["risk_level"] = "Increased"
        elif repetition_count > 1:
            context["conclusion"] = "Recurring pothole — likely worsening, needs prioritized repair"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Pothole detected, standard road hazard"

    elif "crack" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Crack may worsen faster due to water seepage"
            context["risk_level"] = "Increased"
        elif repetition_count > 1:
            context["conclusion"] = "Recurring crack — monitor for expansion"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Standard crack, no weather-related urgency"
    else:
        context["conclusion"] = "General road anomaly detected"

    return context

In [10]:
import ollama

def context_agent_llm(incident, weather, repetition_count=1):
    """
    Uses the rule-based context_agent() for the actual facts/logic,
    then asks a local LLM to phrase it as a natural-language explanation.
    """
    rule_based_result = context_agent(incident, weather, repetition_count)

    prompt = f"""You are a road maintenance assistant. Given this structured incident data, write a single clear, professional sentence explaining the situation for a government dashboard. Do not add information not given below.

Incident ID: {rule_based_result['incident_id']}
Anomaly type: {rule_based_result['anomaly']}
Weather condition: {rule_based_result['weather_condition']}
Times detected: {rule_based_result['repetition_count']}
Preliminary conclusion: {rule_based_result['conclusion']}
Risk level: {rule_based_result['risk_level']}

Write only the explanation sentence, nothing else."""

    response = ollama.chat(model="llama3.2:3b", messages=[
        {"role": "user", "content": prompt}
    ])

    rule_based_result["llm_explanation"] = response["message"]["content"].strip()
    return rule_based_result

In [16]:
# Use your verified incident + a real weather record from Stage 1
test_incident = incidents[0]
test_weather = stage1_frames[0]["weather"]

result = context_agent_llm(test_incident, test_weather, repetition_count=1)
print(result)

{'incident_id': 'RA-0118', 'anomaly': 'D40_pothole', 'weather_condition': 'Clouds', 'repetition_count': 1, 'conclusion': 'Pothole detected, standard road hazard', 'risk_level': 'Normal', 'llm_explanation': 'A pothole has been detected on the road, classified as a standard road hazard with a normal risk level.'}


In [17]:
rainy_weather = {"condition": "Rain", "description": "light rain", "temp_c": 27.5, "humidity": 88}

result_rainy = context_agent_llm(test_incident, rainy_weather, repetition_count=1)
print(result_rainy)

{'incident_id': 'RA-0118', 'anomaly': 'D40_pothole', 'weather_condition': 'Rain', 'repetition_count': 1, 'conclusion': 'Pothole with active rain — increased vehicle hazard', 'risk_level': 'Increased', 'llm_explanation': 'A pothole has been identified on the road, exacerbated by the current rain weather conditions, posing a heightened risk to vehicle safety.'}


In [13]:
result_repeated = context_agent_llm(test_incident, test_weather, repetition_count=3)
print(result_repeated)

{'incident_id': 'RA-0118', 'anomaly': 'D40_pothole', 'weather_condition': 'Clouds', 'repetition_count': 3, 'conclusion': 'Pothole detected, standard road hazard', 'risk_level': 'Normal', 'llm_explanation': 'A pothole has been detected on the road, classified as a standard road hazard, under normal conditions.'}


In [18]:
result_repeated = context_agent_llm(test_incident, test_weather, repetition_count=3)
print(result_repeated)

{'incident_id': 'RA-0118', 'anomaly': 'D40_pothole', 'weather_condition': 'Clouds', 'repetition_count': 3, 'conclusion': 'Recurring pothole — likely worsening, needs prioritized repair', 'risk_level': 'Increased', 'llm_explanation': 'A recurring pothole has been detected on multiple occasions, particularly during cloudy weather conditions, and requires prioritized repair to mitigate the risk of further damage.'}
